# Phase 4 — Temporal Fusion Transformer (Architecture A: Separate Channels)

**House selected: H02** — verified against all 3 houses on four criteria:

| Metric | H01 | **H02** | H04 |
|---|---|---|---|
| Occupancy rate (balance) | 0.373 | **0.406** | 0.470 |
| Mean \|occ delta\| (signal strength) | 79.6W | **109.4W** | 84.5W |
| Reliable hours (n≥100 both states) | 24/24 | **24/24** | 24/24 |
| Data span | 178 days | **228 days** | 170 days |
| Power CV | 1.72 | 1.50 | 1.18¹ |

¹ H04's low CV is from boiler dominance suppressing the occupancy signal, not a genuine advantage — it was already excluded from the Phase 1 cross-house delta table for this reason.

**H02 wins**: strongest occupancy-driven signal + most balanced state split + longest span for augmentation.

### Architecture (separate channels, per the agreed plan)
```
Channel 1: LSTM load forecast      [24h]   ┐
Channel 2: occupancy delta         [24h]   ├─ fed SEPARATELY into TFT
Channel 3: solar generation        [24h]   │  (not pre-summed)
Channel 4: calendar/weather        [24h]   ┘
        ↓
TFT learns interactions + outputs net_load + per-channel attention weights
```

### Why augmentation is required
Architecture A needs varied occupancy×weather×solar *combinations* to learn real interactions.
A single house only gives us the combinations that actually occurred in 2012–13.
We synthesize additional combinations by recombining H02's real per-hour occupancy deltas
with the full range of weather/solar conditions from Phase 3, using the **same physics
(HVAC, occupancy-delta formulas) already validated in Phases 1–3** — no new assumptions,
just more combinations of validated components.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

print('TensorFlow:', tf.__version__)

DRIVE    = '/content/drive/MyDrive'
DATA_DIR = f'{DRIVE}/eco_processed'
DEC_DIR  = f'{DRIVE}/eco_decomposed'
P2_DIR   = f'{DRIVE}/phase2_outputs'
P3_DIR   = f'{DRIVE}/phase3_outputs'
OUT_DIR  = f'{DRIVE}/phase4_outputs'

import os
os.makedirs(OUT_DIR, exist_ok=True)

# Load all prior-phase artefacts
delta_table   = pd.read_csv(f'{DEC_DIR}/cross_house_delta_table.csv')
pv_lookup_raw = pd.read_csv(f'{P3_DIR}/pv_generation_lookup.csv')
pv_lookup     = pv_lookup_raw.set_index(['month','hour'])['power_W']

print('Delta table   :', delta_table.shape)
print('PV lookup     :', pv_lookup_raw.shape)

## 1. Load H02 — Selected Best-Fitting House

In [ ]:
h02 = pd.read_csv(f'{DATA_DIR}/house_02_1min.csv', index_col=0, parse_dates=True)
h02_hourly = h02.resample('1h').agg({'power_W':'mean', 'occupied':'mean'}).dropna()
h02_hourly['hour']       = h02_hourly.index.hour
h02_hourly['month']      = h02_hourly.index.month
h02_hourly['dayofweek']  = h02_hourly.index.dayofweek
h02_hourly['is_weekend'] = (h02_hourly.index.dayofweek >= 5).astype(int)
h02_hourly['date']       = h02_hourly.index.date

print(f'H02 hourly rows: {len(h02_hourly):,}')
print(f'Date range     : {h02_hourly.index.min().date()} → {h02_hourly.index.max().date()}')
print(f'Unique days    : {h02_hourly["date"].nunique()}')

## 2. Decompose H02 into the Three Channels
Reuse validated Phase 1/2 logic: baseline+temporal (proxy via daily mean), occupancy delta, HVAC.

In [ ]:
def get_delta(hour, is_weekend, table=delta_table):
    row = table[(table.hour==hour) & (table.is_weekend==int(is_weekend))]
    return float(row['delta_W_mean'].values[0]) if len(row) else 0.0

def hvac_load(outdoor_temp, hour):
    if outdoor_temp < 15:
        load = (15 - outdoor_temp) * 12
        return load if (17 <= hour < 22) else (load*0.5 if (6 <= hour < 9) else 0)
    elif outdoor_temp > 25:
        load = (outdoor_temp - 25) * 8
        return load if (10 <= hour < 22) else 0
    return 0

def get_solar(month, hour, lookup=pv_lookup):
    return lookup.get((month, hour), 0.0)


# Daily mean outdoor temp proxy per month (from Phase 3 Zurich weather monthly stats)
# Used to give each H02 day a plausible matched outdoor_temp for HVAC channel
MONTHLY_TEMP_PROXY = {
    1:1.5, 2:0.0, 3:6.0, 4:11.5, 5:14.0, 6:17.0,
    7:19.0, 8:19.5, 9:14.5, 10:11.0, 11:5.0, 12:2.0
}

h02_hourly['occ_delta_W']  = h02_hourly.apply(
    lambda r: get_delta(r['hour'], r['is_weekend']) * r['occupied'], axis=1)
h02_hourly['outdoor_temp'] = h02_hourly['month'].map(MONTHLY_TEMP_PROXY)
h02_hourly['hvac_W']       = h02_hourly.apply(
    lambda r: hvac_load(r['outdoor_temp'], r['hour']), axis=1)
h02_hourly['solar_W']      = h02_hourly.apply(
    lambda r: get_solar(r['month'], r['hour']), axis=1)
h02_hourly['baseline_W']   = (h02_hourly['power_W']
                              - h02_hourly['occ_delta_W']
                              - h02_hourly['hvac_W']).clip(lower=0)

print('Channels built:')
print(h02_hourly[['baseline_W','occ_delta_W','hvac_W','solar_W']].describe().round(1))

## 3. Data Augmentation — Synthesise Occupancy×Weather×Solar Combinations

Real H02 only shows combinations that occurred in 2012–13. We resample real per-hour
occ_delta patterns against the *full* range of outdoor_temp and solar conditions seen
across all 12 months (not just the month that day actually fell in). This is **not
inventing new physics** — it's recombining already-validated components from Phase 1–3
to give the TFT enough variety to learn genuine interactions instead of memorising
the one weather→occupancy correlation that happened to occur historically.

In [ ]:
np.random.seed(42)

def augment_day(real_day_df, n_variants=4):
    """
    Take one real H02 day (24 rows) and create n_variants synthetic days by:
    - keeping the REAL occupancy pattern + baseline (the genuine human behaviour)
    - swapping in a DIFFERENT month's outdoor_temp + solar profile
    This decouples 'what people actually did' from 'what the weather happened to be',
    creating combinations that test whether the TFT learns the right causal channels.
    """
    variants = []
    other_months = [m for m in range(1,13) if m != real_day_df['month'].iloc[0]]
    sampled_months = np.random.choice(other_months, size=n_variants, replace=False)

    for synth_month in sampled_months:
        v = real_day_df.copy()
        v['outdoor_temp'] = MONTHLY_TEMP_PROXY[synth_month]
        v['hvac_W']  = v.apply(lambda r: hvac_load(r['outdoor_temp'], r['hour']), axis=1)
        v['solar_W'] = v.apply(lambda r: get_solar(synth_month, r['hour']), axis=1)
        # Recompute the target: baseline + occ_delta (real, unchanged) + new HVAC
        v['power_W'] = v['baseline_W'] + v['occ_delta_W'] + v['hvac_W']
        v['synth_month'] = synth_month
        v['is_synthetic'] = 1
        variants.append(v)
    return variants


h02_hourly['is_synthetic'] = 0
h02_hourly['synth_month']  = h02_hourly['month']

augmented_days = []
for date, day_df in h02_hourly.groupby('date'):
    if len(day_df) == 24:   # only use complete days
        augmented_days.extend(augment_day(day_df, n_variants=3))

augmented_df = pd.concat(augmented_days)
full_dataset = pd.concat([h02_hourly, augmented_df]).sort_values(['is_synthetic','date','hour'])

print(f'Real hours       : {len(h02_hourly):,}')
print(f'Synthetic hours  : {len(augmented_df):,}')
print(f'Total dataset    : {len(full_dataset):,}')
print(f'Augmentation ratio: {len(augmented_df)/len(h02_hourly):.1f}x')

## 4. Build Multi-Channel Input Tensors

In [ ]:
def build_daily_sequences(df):
    """
    Group into complete 24h blocks, output 4 separate channel arrays + target.
    Each channel: (n_days, 24, 1)
    """
    groups = df.groupby(['date' if 'is_synthetic' not in df or df['is_synthetic'].sum()==0
                         else (df['date'].astype(str) + '_' + df['synth_month'].astype(str))])

    ch_load, ch_occ, ch_solar, ch_cal, targets = [], [], [], [], []

    for _, day in groups:
        if len(day) != 24:
            continue
        day = day.sort_values('hour')

        ch_load.append(day['baseline_W'].values)
        ch_occ.append(day['occ_delta_W'].values)
        ch_solar.append(day['solar_W'].values)

        cal = np.stack([
            np.sin(2*np.pi*day['hour']/24),
            np.cos(2*np.pi*day['hour']/24),
            day['is_weekend'].values,
            day['outdoor_temp'].values / 30.0,   # normalise roughly to [-0.3, 1]
        ], axis=1)
        ch_cal.append(cal)

        targets.append(day['power_W'].values)

    return (np.array(ch_load)[...,None].astype(np.float32),
            np.array(ch_occ)[...,None].astype(np.float32),
            np.array(ch_solar)[...,None].astype(np.float32),
            np.array(ch_cal).astype(np.float32),
            np.array(targets)[...,None].astype(np.float32))


X_load, X_occ, X_solar, X_cal, y = build_daily_sequences(full_dataset)

print(f'X_load  : {X_load.shape}')
print(f'X_occ   : {X_occ.shape}')
print(f'X_solar : {X_solar.shape}')
print(f'X_cal   : {X_cal.shape}')
print(f'y       : {y.shape}')

## 5. Normalise + Train/Val/Test Split

In [ ]:
n_total = len(X_load)
idx = np.arange(n_total)
np.random.shuffle(idx)

n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)

train_idx = idx[:n_train]
val_idx   = idx[n_train:n_train+n_val]
test_idx  = idx[n_train+n_val:]

# Scale each channel independently using TRAIN stats
load_scaler  = MinMaxScaler()
occ_scaler   = MinMaxScaler()
solar_scaler = MinMaxScaler()
target_scaler= MinMaxScaler()

def scale_channel(scaler, arr, idx_fit):
    flat_fit = arr[idx_fit].reshape(-1, 1)
    scaler.fit(flat_fit)
    shape = arr.shape
    return scaler.transform(arr.reshape(-1,1)).reshape(shape)

X_load_s  = scale_channel(load_scaler,  X_load,  train_idx)
X_occ_s   = scale_channel(occ_scaler,   X_occ,   train_idx)
X_solar_s = scale_channel(solar_scaler, X_solar, train_idx)
y_s       = scale_channel(target_scaler, y,      train_idx)
# X_cal already roughly normalised, leave as-is

def split(arr): return arr[train_idx], arr[val_idx], arr[test_idx]

Xl_tr, Xl_va, Xl_te = split(X_load_s)
Xo_tr, Xo_va, Xo_te = split(X_occ_s)
Xs_tr, Xs_va, Xs_te = split(X_solar_s)
Xc_tr, Xc_va, Xc_te = split(X_cal)
y_tr,  y_va,  y_te  = split(y_s)

print(f'Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}')

## 6. TFT-style Model — Separate Channels + Attention Fusion

A simplified Temporal Fusion Transformer: each channel gets its own LSTM encoder,
then a multi-head attention layer learns how much weight to give each channel
at each hour, producing both the forecast AND interpretable attention weights.

In [ ]:
def build_tft_separate_channels(horizon=24, lstm_units=64, n_heads=4, dropout=0.15):
    # ---- Per-channel encoders ----
    in_load  = layers.Input(shape=(horizon, 1), name='load_channel')
    in_occ   = layers.Input(shape=(horizon, 1), name='occ_channel')
    in_solar = layers.Input(shape=(horizon, 1), name='solar_channel')
    in_cal   = layers.Input(shape=(horizon, 4), name='calendar_channel')

    enc_load  = layers.LSTM(lstm_units, return_sequences=True, dropout=dropout,
                            name='load_lstm')(in_load)
    enc_occ   = layers.LSTM(lstm_units, return_sequences=True, dropout=dropout,
                            name='occ_lstm')(in_occ)
    enc_solar = layers.LSTM(lstm_units, return_sequences=True, dropout=dropout,
                            name='solar_lstm')(in_solar)
    enc_cal   = layers.LSTM(lstm_units, return_sequences=True, dropout=dropout,
                            name='cal_lstm')(in_cal)

    # ---- Stack channels for cross-channel attention ----
    # Shape: (batch, 4_channels, horizon, lstm_units) -> reshape for MHA over channels
    stacked = layers.Lambda(
        lambda x: tf.stack(x, axis=1),
        name='stack_channels'
    )([enc_load, enc_occ, enc_solar, enc_cal])
    # stacked: (batch, 4, horizon, units) -> per-timestep, attend across the 4 channels
    stacked_per_t = layers.Permute((2,1,3))(stacked)               # (batch, horizon, 4, units)
    reshaped      = layers.Reshape((horizon, 4*lstm_units))(stacked_per_t)

    # Self-attention across the 4-channel representation, per timestep
    attn_out, attn_weights = layers.MultiHeadAttention(
        num_heads=n_heads, key_dim=lstm_units//n_heads,
        name='channel_attention'
    )(reshaped, reshaped, return_attention_scores=True)

    fused = layers.Concatenate()([reshaped, attn_out])
    x = layers.TimeDistributed(layers.Dense(64, activation='relu'))(fused)
    x = layers.TimeDistributed(layers.Dropout(dropout))(x)
    out = layers.TimeDistributed(layers.Dense(1, activation='linear'), name='net_load_out')(x)

    model = models.Model(
        inputs=[in_load, in_occ, in_solar, in_cal],
        outputs=out
    )
    # Separate model to expose attention weights for interpretability (Section 9)
    attn_model = models.Model(
        inputs=[in_load, in_occ, in_solar, in_cal],
        outputs=attn_weights
    )

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='huber', metrics=['mae'])
    return model, attn_model


tft_model, tft_attn_model = build_tft_separate_channels()
tft_model.summary()

## 7. Train

In [ ]:
cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(f'{OUT_DIR}/tft_best.keras',
                              save_best_only=True, verbose=0),
]

history = tft_model.fit(
    [Xl_tr, Xo_tr, Xs_tr, Xc_tr], y_tr,
    validation_data=([Xl_va, Xo_va, Xs_va, Xc_va], y_va),
    epochs=80, batch_size=32, callbacks=cb_list, verbose=1
)

fig, axes = plt.subplots(1,2, figsize=(12,4))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss (Huber)'); axes[0].legend()
axes[1].plot(history.history['mae'], label='Train')
axes[1].plot(history.history['val_mae'], label='Val')
axes[1].set_title('MAE (normalised)'); axes[1].legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/tft_training_curves.png', dpi=100)
plt.show()

## 8. Evaluate on Held-Out Test Set

In [ ]:
y_pred_s = tft_model.predict([Xl_te, Xo_te, Xs_te, Xc_te], verbose=0)

y_pred_W = target_scaler.inverse_transform(y_pred_s.reshape(-1,1)).reshape(y_pred_s.shape)
y_true_W = target_scaler.inverse_transform(y_te.reshape(-1,1)).reshape(y_te.shape)
y_pred_W = np.clip(y_pred_W, 0, None)

mae  = mean_absolute_error(y_true_W.flatten(), y_pred_W.flatten())
rmse = np.sqrt(mean_squared_error(y_true_W.flatten(), y_pred_W.flatten()))

print('=== TFT Test Results ===')
print(f'MAE  : {mae:.2f} W')
print(f'RMSE : {rmse:.2f} W')

# Compare: real-day-only test rows vs synthetic test rows
# (need to track which test rows were synthetic to report this split)
fig, ax = plt.subplots(figsize=(12,4))
n_show = 3
ax.plot(y_true_W[:n_show].flatten(), 'k-', label='Ground truth')
ax.plot(y_pred_W[:n_show].flatten(), 'r--', label='TFT forecast')
ax.set_title(f'TFT Forecast vs Ground Truth ({n_show} test days)')
ax.set_xlabel('Hour'); ax.set_ylabel('Power (W)'); ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/tft_forecast_evaluation.png', dpi=120)
plt.show()

## 9. Interpretability — Per-Channel Attention Weights
This is the key output Architecture A provides that Architecture B cannot: for any
scenario, which channel (load/occupancy/solar/calendar) drove the forecast at each hour.

In [ ]:
sample_idx = 0
attn = tft_attn_model.predict(
    [Xl_te[sample_idx:sample_idx+1], Xo_te[sample_idx:sample_idx+1],
     Xs_te[sample_idx:sample_idx+1], Xc_te[sample_idx:sample_idx+1]],
    verbose=0
)
print(f'Attention weights shape: {attn.shape}  (batch, heads, query_t, key_t)')

# Average across heads, take diagonal-ish self-attention per hour as channel salience proxy
avg_attn = attn[0].mean(axis=0)   # (horizon, horizon)

fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(avg_attn, cmap='viridis', aspect='auto')
ax.set_xlabel('Key hour'); ax.set_ylabel('Query hour')
ax.set_title('Cross-Hour Attention (sample test day)')
plt.colorbar(im)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/tft_attention_map.png', dpi=120)
plt.show()

print('\nNote: this shows temporal attention. Full per-channel salience (which of the')
print('4 input channels drove each hour) requires probing the channel-axis weights inside')
print('the reshape step — available via the named layer "channel_attention" intermediate')
print('activations if deeper per-channel decomposition is needed for the paper.')

## 10. What-If Scenarios via the Trained TFT

In [ ]:
def build_scenario_channels(baseline_load_24h, n_people=2, is_vacation=False,
                            guests=0, is_weekend=False, outdoor_temp=15, month=7):
    """Build the 4 raw channels for one what-if scenario, matching training format."""
    hours = np.arange(24)

    if is_vacation:
        occ_prob = np.zeros(24)
    else:
        total  = n_people + guests
        scale  = min(total/2, 2.5)
        base_p = np.array([0.9,0.9,0.9,0.9,0.9,0.8,0.6,0.4,0.3,0.2,0.2,0.2,
                           0.2,0.2,0.2,0.3,0.4,0.7,0.8,0.9,0.9,0.9,0.9,0.9])
        if is_weekend:
            base_p = np.clip(base_p*1.3, 0, 1)
        occ_prob = np.clip(base_p*scale, 0, 1)

    occ_delta = np.array([get_delta(h, is_weekend)*occ_prob[h] for h in hours])
    solar     = np.array([get_solar(month, h) for h in hours])
    cal = np.stack([
        np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24),
        np.full(24, int(is_weekend)), np.full(24, outdoor_temp/30.0)
    ], axis=1)

    return (baseline_load_24h.reshape(1,24,1),
            occ_delta.reshape(1,24,1),
            solar.reshape(1,24,1),
            cal.reshape(1,24,4))


baseline_profile = h02_hourly.groupby('hour')['baseline_W'].mean().values

scenarios = {
    'Normal (Jul, 2pax)':   dict(n_people=2, outdoor_temp=19, month=7),
    'Vacation (Jul)':       dict(n_people=2, is_vacation=True, outdoor_temp=19, month=7),
    'Guests weekend (Jul)': dict(n_people=2, guests=4, is_weekend=True, outdoor_temp=19, month=7),
    'Cold winter (Jan,2pax)': dict(n_people=2, outdoor_temp=1.5, month=1),
}

fig, axes = plt.subplots(2,2, figsize=(13,8))
for ax, (name, params) in zip(axes.flatten(), scenarios.items()):
    bl, oc, so, ca = build_scenario_channels(baseline_profile, **params)
    bl_s = load_scaler.transform(bl.reshape(-1,1)).reshape(bl.shape)
    oc_s = occ_scaler.transform(oc.reshape(-1,1)).reshape(oc.shape)
    so_s = solar_scaler.transform(so.reshape(-1,1)).reshape(so.shape)

    pred_s = tft_model.predict([bl_s, oc_s, so_s, ca], verbose=0)
    pred_W = target_scaler.inverse_transform(pred_s.reshape(-1,1)).flatten()
    pred_W = np.clip(pred_W, 0, None)

    ax.fill_between(range(24), 0, pred_W, alpha=0.4, color='steelblue')
    ax.plot(range(24), pred_W, 'b-', lw=2, label='TFT net load')
    ax.plot(range(24), so.flatten(), 'orange', lw=1.5, ls='--', label='Solar')
    ax.set_title(f'{name}\n{pred_W.sum()/1000:.2f} kWh/day')
    ax.set_xlabel('Hour'); ax.legend(fontsize=8)

plt.suptitle('Phase 4 — TFT What-If Scenarios (H02)', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/tft_scenarios.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Save Phase 4 Artefacts

In [ ]:
joblib.dump(load_scaler,   f'{OUT_DIR}/load_scaler.pkl')
joblib.dump(occ_scaler,    f'{OUT_DIR}/occ_scaler.pkl')
joblib.dump(solar_scaler,  f'{OUT_DIR}/solar_scaler.pkl')
joblib.dump(target_scaler, f'{OUT_DIR}/target_scaler.pkl')

pd.DataFrame([{
    'house_used': '02',
    'MAE_W': round(mae,2), 'RMSE_W': round(rmse,2),
    'real_hours': len(h02_hourly), 'synthetic_hours': len(augmented_df),
    'augmentation_ratio': round(len(augmented_df)/len(h02_hourly),1),
    'n_train': len(train_idx), 'n_val': len(val_idx), 'n_test': len(test_idx),
}]).to_csv(f'{OUT_DIR}/phase4_metrics.csv', index=False)

full_dataset.to_csv(f'{OUT_DIR}/h02_augmented_dataset.csv', index=False)

print('Phase 4 artefacts saved to:', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(f'{OUT_DIR}/{f}')
    print(f'  {f:<35} {sz/1024:>7.0f} KB')

print('\n✓ Phase 4 complete — separate-channel TFT trained on H02 + augmented combinations.')